[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/alignment_2/alignment_metrics.ipynb)

In [ ]:
import os
import sys

try:
    import google.colab

    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop
    ! pip install parangonar
    ! pip install pymatchmaker==0.4.0

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone https://github.com/MusicalInformatics/miws26.git
    sys.path.insert(0, "./miws26/alignment_1/")
    MEC_VARIATION_DIR = "./miws26/alignment_1/mec-variation"
else:
    # The notebook is in the alignment_2 folder. We use the code and the data of the alignment_1 folder
    ALIGNMENT_1_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "alignment_1")
    sys.path.insert(0, ALIGNMENT_1_DIR)
    MEC_VARIATION_DIR = os.path.join(ALIGNMENT_1_DIR, "mec-variation")

# Evaluating Music Alignments

How do we know if an alignment is *good*? In this notebook we look at the metrics that are used to evaluate music alignment algorithms. We start with the case of **offline, symbolic, note-level alignment** between a score and a performance, which is evaluated with the **F-score**. This is the metric used, e.g., to evaluate the score-to-performance alignments of the ASAP dataset:

> Silvan Peter, Carlos Cancino-Chacón, Francesco Foscarin, Andrew McLeod, Florian Henkel, Emmanouil Karystinaios, and Gerhard Widmer. *Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset*. Transactions of the International Society for Music Information Retrieval, 6(1), 2023.

In the second part (Section 7), we will look at the metrics for *online* alignment (score following), which are included in Matchmaker.

We will use the `mec-variation` dataset, which contains several performances of the same score together with *ground truth* note-level alignments.

The code to compute the F-score is included in [Parangonar](https://github.com/sildater/parangonar), the package that we also use for note-level alignment (`parangonar.evaluate`). The same function is available in the helper module of the `alignment_1` folder. The metrics for online alignment live in [Matchmaker](https://github.com/pymatchmaker/matchmaker) (`matchmaker.utils.eval`), and we use them in Section 7.

In [ ]:
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch, Rectangle
import pandas as pd
import partitura as pt

from alignment import fast_dynamic_time_warping, greedy_note_alignment
from helper import dummy_linear_alignment
from parangonar.evaluate import fscore_alignments

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format ='retina'

MUSICXML_DIR = os.path.join(MEC_VARIATION_DIR, "scores", "musicxml")
MATCH_DIR = os.path.join(MEC_VARIATION_DIR, "alignments", "match")

## 1. Note-Level Alignments

A note-level alignment links the notes of a score to the notes of a performance. It is represented as a list of dictionaries, each of which has a `label`:

* `match`: a score note (`score_id`) was played as a performed note (`performance_id`).
* `deletion`: a score note (`score_id`) was **not played** (there is no performed note for it).
* `insertion`: a performed note (`performance_id`) is **not in the score** (e.g., a wrong note or an ornament).

Let's load a score, and a performance with its ground truth alignment. The `.match` file contains the performance and the alignment. (As in the `Symbolic_Music_Alignment` notebook, we unfold the score so that the ids of the notes match the ones in the alignment.)

In [ ]:
score_fn = os.path.join(MUSICXML_DIR, "short.musicxml")
score = pt.load_score(score_fn)
# The alignments of this dataset refer to the "unfolded" score (i.e., with the repeats written out)
score = pt.score.unfold_part_maximal(score[0])
snote_array = score.note_array()

match_fn = os.path.join(MATCH_DIR, "short1.match")
performance, gt_alignment = pt.load_match(
    match_fn, pedal_threshold=127, first_note_at_zero=True
)
pnote_array = performance.note_array()

print(f"The score has {len(snote_array)} notes")
print(f"The performance has {len(pnote_array)} notes")
gt_alignment[:3]

In [ ]:
from collections import Counter

Counter(a["label"] for a in gt_alignment)

Most notes in this performance are matches. There is also one *insertion*, i.e., a performed note that is not in the score. Let's see which one it is:

In [ ]:
for a in gt_alignment:
    if a["label"] != "match":
        print(a)

## 2. Visualizing an Alignment

It is easier to understand an alignment if we *see* it. The function below plots the piano roll of the score (top) and the piano roll of the performance (bottom), and draws a line between the notes that are aligned. Notes that are not aligned (deletions in the score, insertions in the performance) are drawn in orange.

We can also give the function a *prediction* and a *ground truth* alignment to compare them (we will use this later).

In [ ]:
CORRECT_COLOR = "tab:green"
WRONG_COLOR = "tab:red"
MISSED_COLOR = "tab:blue"
UNALIGNED_COLOR = "darkorange"


def alignment_to_sets(
    alignment: List[dict],
) -> Tuple[Set[Tuple[str, str]], Set[str], Set[str]]:
    """Split an alignment into the set of matched (score_id, performance_id) pairs, the deleted score notes and the inserted performance notes"""
    matches = {
        (str(a["score_id"]), str(a["performance_id"]))
        for a in alignment
        if a["label"] == "match"
    }
    deletions = {str(a["score_id"]) for a in alignment if a["label"] == "deletion"}
    insertions = {str(a["performance_id"]) for a in alignment if a["label"] == "insertion"}
    return matches, deletions, insertions


def draw_piano_roll(ax, note_array, onset_key, duration_key, color, unaligned_ids, selected_ids):
    """Draw the notes as rectangles"""
    for note in note_array:
        if str(note["id"]) not in selected_ids:
            continue
        ax.add_patch(
            Rectangle(
                (note[onset_key], note["pitch"] - 0.4),
                max(note[duration_key], 0.05),
                0.8,
                facecolor=UNALIGNED_COLOR if str(note["id"]) in unaligned_ids else color,
                edgecolor="none",
            )
        )


def plot_note_alignment(
    prediction: List[dict],
    ground_truth: List[dict] = None,
    beat_range: Tuple[float, float] = None,
    title: str = None,
    figsize: Tuple[float, float] = (14, 6),
):
    """
    Plot a note-level alignment between the score (top) and the performance (bottom).

    If only `prediction` is given, all the pairs of matched notes are drawn in gray.
    If `ground_truth` is also given, the pairs are colored:
    green: in the prediction and in the ground truth (correct)
    red: only in the prediction (wrong)
    blue (dashed): only in the ground truth (missed)

    beat_range: only plot the notes of the score in this range (in beats), and the
        performed notes that are close to the notes aligned to them
    """
    pred_matches, pred_del, pred_ins = alignment_to_sets(prediction)
    if ground_truth is None:
        gt_matches, gt_del, gt_ins = set(), set(), set()
    else:
        gt_matches, gt_del, gt_ins = alignment_to_sets(ground_truth)
    all_matches = pred_matches | gt_matches

    score_ids = {str(n["id"]) for n in snote_array}
    perf_ids = {str(n["id"]) for n in pnote_array}
    onset_beat = {str(n["id"]): n["onset_beat"] for n in snote_array}
    onset_sec = {str(n["id"]): n["onset_sec"] for n in pnote_array}
    pitch_of = {
        str(n["id"]): n["pitch"] for n in list(snote_array) + list(pnote_array)
    }

    # Select the notes that we plot
    if beat_range is not None:
        score_ids = {i for i in score_ids if beat_range[0] <= onset_beat[i] < beat_range[1]}
        partners = {p for s, p in all_matches if s in score_ids}
        t0 = min(onset_sec[p] for p in partners) - 0.3
        t1 = max(onset_sec[p] for p in partners) + 0.3
        perf_ids = {i for i in perf_ids if t0 <= onset_sec[i] <= t1}
        beat_lim = beat_range
        sec_lim = (t0, t1)
    else:
        beat_lim = (0, snote_array["onset_beat"].max() + 1)
        sec_lim = (0, pnote_array["onset_sec"].max() + 0.5)

    fig, (ax_s, ax_p) = plt.subplots(
        2, 1, figsize=figsize, gridspec_kw={"hspace": 0.55}
    )
    draw_piano_roll(
        ax_s, snote_array, "onset_beat", "duration_beat", "0.45",
        pred_del | gt_del, score_ids,
    )
    draw_piano_roll(
        ax_p, pnote_array, "onset_sec", "duration_sec", "0.45",
        pred_ins | gt_ins, perf_ids,
    )

    # Draw the lines between the aligned notes
    for s, p in sorted(all_matches):
        if s not in score_ids or p not in perf_ids:
            continue
        if ground_truth is None:
            color, style, width, zorder = "0.2", "-", 0.7, 2
        elif (s, p) in pred_matches and (s, p) in gt_matches:
            color, style, width, zorder = CORRECT_COLOR, "-", 0.8, 2
        elif (s, p) in pred_matches:
            color, style, width, zorder = WRONG_COLOR, "-", 1.6, 4
        else:
            color, style, width, zorder = MISSED_COLOR, "--", 1.6, 3
        fig.add_artist(
            ConnectionPatch(
                xyA=(onset_beat[s], pitch_of[s] - 0.4), coordsA=ax_s.transData,
                xyB=(onset_sec[p], pitch_of[p] + 0.4), coordsB=ax_p.transData,
                color=color, linestyle=style, linewidth=width, zorder=zorder,
            )
        )

    pitches = [pitch_of[i] for i in score_ids | perf_ids]
    for ax, xlim, xlabel in [
        (ax_s, beat_lim, "Score time (beats)"),
        (ax_p, sec_lim, "Performance time (s)"),
    ]:
        ax.set_xlim(xlim)
        ax.set_ylim(min(pitches) - 2, max(pitches) + 2)
        ax.set_xlabel(xlabel)
    ax_s.set_ylabel("Score\nMIDI pitch")
    ax_p.set_ylabel("Performance\nMIDI pitch")
    # score axis on top, so that the labels are not crossed by the lines between the notes
    ax_s.xaxis.tick_top()
    ax_s.xaxis.set_label_position("top")

    handles = [Line2D([], [], color="0.2", label="aligned") if ground_truth is None else
               Line2D([], [], color=CORRECT_COLOR, label="correct (in prediction and ground truth)")]
    if ground_truth is not None:
        handles += [
            Line2D([], [], color=WRONG_COLOR, linewidth=1.6, label="wrong (only in prediction)"),
            Line2D([], [], color=MISSED_COLOR, linestyle="--", linewidth=1.6, label="missed (only in ground truth)"),
        ]
    handles += [mpatches.Patch(color=UNALIGNED_COLOR, label="note without partner (deletion / insertion)")]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
    fig.suptitle(title if title is not None else "")
    fig.subplots_adjust(left=0.07, right=0.99, top=0.88, bottom=0.14)
    return fig

In [ ]:
fig = plot_note_alignment(gt_alignment, title="Ground truth alignment")
plt.show()

There are too many notes to see the details, so let's zoom in on a few beats. Use the slider to move through the piece (and try to find the insertion!).

In [ ]:
import ipywidgets as widgets

max_beat = snote_array["onset_beat"].max()


def plot_gt_window(start_beat, length):
    fig = plot_note_alignment(
        gt_alignment,
        beat_range=(start_beat, start_beat + length),
        title="Ground truth alignment",
        figsize=(12, 5.5),
    )
    plt.show()


widgets.interact(
    plot_gt_window,
    start_beat=widgets.FloatSlider(value=0, min=0, max=max_beat, step=1, description="Beat", continuous_update=False),
    length=widgets.FloatSlider(value=8, min=2, max=16, step=1, description="Length", continuous_update=False),
);

## 3. A Predicted Alignment

To have something to evaluate, we need an alignment computed by an algorithm. We will use the offline alignment methods from `Symbolic_Music_Alignment` (see that notebook for the details): dynamic time warping (DTW) of the features of the score and the performance, followed by a greedy matching of the notes. We compare three methods:

1. **Linear**: assumes that the performance has a constant tempo (a straight line, no actual alignment)
2. **DTW (pitch classes)**: DTW using pitch class distributions
3. **DTW (piano roll)**: DTW using piano rolls

In [ ]:
def compute_alignment(method: str, score, performance) -> List[dict]:
    """
    Compute a note-level alignment between a score and a performance.

    method: "linear", "DTW (pitch class)" or "DTW (piano roll)"
    """
    snote_array = score.note_array()
    pnote_array = performance.note_array()
    if method == "DTW (pitch class)":
        score_feat, sidx = pt.utils.music.compute_pitch_class_pianoroll(
            score, time_unit="beat", time_div=8, return_idxs=True,
            binary=True, note_separation=True,
        )
        perf_feat, pidx = pt.utils.music.compute_pitch_class_pianoroll(
            performance, time_unit="sec", time_div=8, return_idxs=True,
            binary=True, note_separation=True,
        )
        reference_features, performance_features = score_feat.T, perf_feat.T
    else:
        score_feat, sidx = pt.utils.music.compute_pianoroll(
            score, time_unit="beat", time_div=8, return_idxs=True,
            piano_range=True, binary=True, note_separation=True,
        )
        perf_feat, pidx = pt.utils.music.compute_pianoroll(
            performance, time_unit="sec", time_div=8, return_idxs=True,
            piano_range=True, binary=True, note_separation=True,
        )
        reference_features = score_feat.toarray().T
        performance_features = perf_feat.toarray().T

    if method == "linear":
        warping_path = dummy_linear_alignment(reference_features, performance_features)
    else:
        warping_path = fast_dynamic_time_warping(
            X=reference_features, Y=performance_features, metric="cityblock"
        )

    return greedy_note_alignment(
        warping_path=warping_path,
        idx1=sidx, note_array1=snote_array,
        idx2=pidx, note_array2=pnote_array,
    )


METHODS = ["linear", "DTW (pitch class)", "DTW (piano roll)"]
predictions = {m: compute_alignment(m, score, performance) for m in METHODS}

for m, pred in predictions.items():
    print(m, dict(Counter(a["label"] for a in pred)))

## 4. Comparing a Prediction with the Ground Truth

An alignment is a *set of pairs*. To evaluate it, we compare the set of pairs in the **prediction** with the set of pairs in the **ground truth**:

* **True positives (TP)**: pairs that are in both the prediction and the ground truth.
* **False positives (FP)**: pairs that are in the prediction, but **not** in the ground truth (the algorithm predicted a wrong pair).
* **False negatives (FN)**: pairs that are in the ground truth, but **not** in the prediction (the algorithm missed a pair).

Let's visualize this for the DTW (piano roll) alignment: green lines are correct, red lines are wrong, and dashed blue lines are the pairs that the algorithm missed.

In [ ]:
fig = plot_note_alignment(
    predictions["DTW (piano roll)"], gt_alignment, title="DTW (piano roll) vs. ground truth"
)
plt.show()

In [ ]:
def plot_pred_window(method, start_beat, length):
    fig = plot_note_alignment(
        predictions[method], gt_alignment,
        beat_range=(start_beat, start_beat + length),
        title=f"{method} vs. ground truth",
        figsize=(12, 5.5),
    )
    plt.show()


widgets.interact(
    plot_pred_window,
    method=widgets.Dropdown(options=METHODS, value="DTW (piano roll)", description="Method"),
    start_beat=widgets.FloatSlider(value=0, min=0, max=max_beat, step=1, description="Beat", continuous_update=False),
    length=widgets.FloatSlider(value=8, min=2, max=16, step=1, description="Length", continuous_update=False),
);

**Question**: Where do the wrong pairs of the linear alignment happen? Why? Compare it to the DTW alignments.

Note that a pair is *either* correct or wrong: it does not matter how far the predicted partner is from the true one.

### Computing the F-score

From the counts of TP, FP and FN, we compute

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{TP}{\#\text{predicted pairs}} \qquad \text{Recall} = \frac{TP}{TP + FN} = \frac{TP}{\#\text{ground truth pairs}}$$

$$\text{F-score} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

* **Precision**: of all the pairs that the algorithm predicted, which proportion is correct?
* **Recall**: of all the pairs in the ground truth, which proportion did the algorithm find?
* **F-score**: the harmonic mean of precision and recall. It is high only if both are high.

Let's compute it step by step for the matches of one of the alignments, using the sets of pairs:

In [ ]:
pred_matches, pred_deletions, pred_insertions = alignment_to_sets(predictions["DTW (piano roll)"])
gt_matches, gt_deletions, gt_insertions = alignment_to_sets(gt_alignment)

true_positives = pred_matches & gt_matches
false_positives = pred_matches - gt_matches
false_negatives = gt_matches - pred_matches

n_tp, n_fp, n_fn = len(true_positives), len(false_positives), len(false_negatives)
precision = n_tp / (n_tp + n_fp)
recall = n_tp / (n_tp + n_fn)
f_score = 2 * precision * recall / (precision + recall)

print(f"TP = {n_tp}, FP = {n_fp}, FN = {n_fn}")
print(f"Precision = {precision:.4f}, Recall = {recall:.4f}, F-score = {f_score:.4f}")

Here is what these numbers mean. The bars show the pairs of the ground truth (top) and of the prediction (bottom):

In [ ]:
def plot_tp_fp_fn(n_tp: int, n_fp: int, n_fn: int, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 2.6))
    # ground truth = TP + FN, prediction = TP + FP
    ax.barh(1, n_tp, color=CORRECT_COLOR, label="TP (correct)")
    ax.barh(1, n_fn, left=n_tp, color=MISSED_COLOR, label="FN (missed)")
    ax.barh(0, n_tp, color=CORRECT_COLOR)
    ax.barh(0, n_fp, left=n_tp, color=WRONG_COLOR, label="FP (wrong)")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Prediction", "Ground truth"])
    ax.set_xlabel("Number of pairs")
    ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.75), ncol=3, frameon=False)
    return ax


plot_tp_fp_fn(
    n_tp, n_fp, n_fn,
    title=f"Precision = TP / prediction = {precision:.3f},   Recall = TP / ground truth = {recall:.3f}",
)
plt.tight_layout()
plt.show()

We do not need to implement this ourselves: `parangonar.evaluate.fscore_alignments` computes it. Its argument `types` selects the labels that we want to evaluate:

In [ ]:
precision_pa, recall_pa, f_score_pa = fscore_alignments(
    prediction=predictions["DTW (piano roll)"],
    ground_truth=gt_alignment,
    types=["match"],
)
print(f"Precision = {precision_pa:.4f}, Recall = {recall_pa:.4f}, F-score = {f_score_pa:.4f}")
assert np.allclose([precision, recall, f_score], [precision_pa, recall_pa, f_score_pa])

### What counts as a correct pair?

`fscore_alignments` considers a predicted item correct only if there is an item in the ground truth that is **exactly equal**, i.e., it has the same `label` **and** the same `score_id` **and** the same `performance_id`. Let's try it with a tiny example, in which the performance has 4 notes, and the 4th note is wrong (an insertion), and the score has 5 notes, where the last one was not played (a deletion):

In [ ]:
toy_gt = [
    {"label": "match", "score_id": "s1", "performance_id": "p1"},
    {"label": "match", "score_id": "s2", "performance_id": "p2"},
    {"label": "match", "score_id": "s3", "performance_id": "p3"},
    {"label": "insertion", "performance_id": "p4"},
    {"label": "deletion", "score_id": "s5"},
]
# A prediction that makes some errors
toy_prediction = [
    {"label": "match", "score_id": "s1", "performance_id": "p1"},  # correct
    {"label": "match", "score_id": "s2", "performance_id": "p3"},  # wrong partner
    {"label": "match", "score_id": "s3", "performance_id": "p2"},  # wrong partner
    {"label": "match", "score_id": "s5", "performance_id": "p4"},  # the wrong note was matched
]

for label in ["match", "insertion", "deletion"]:
    p, r, f, n_pred, n_gt = fscore_alignments(toy_prediction, toy_gt, [label], return_numbers=True)
    print(f"{label:10s} predicted: {n_pred}, ground truth: {n_gt} -> P = {p:.2f}, R = {r:.2f}, F = {f:.2f}")

* There is only 1 correct `match` out of 4 predicted (P = 0.25), and out of 3 in the ground truth (R = 0.33).
* The algorithm predicted no `insertion` and no `deletion`. Since there are such items in the ground truth, the recall is 0. When nothing is predicted, the precision is defined to be 0.
* If there is **nothing** in the prediction **and** nothing in the ground truth for a label (e.g., a performance without any wrong notes, and an algorithm that does not predict any), the F-score is defined to be 1.

### The F-score for each type

Alignments contain three kinds of items, and it is usual to report the F-score of **each type** separately (and sometimes of all of them together, `types=["match", "insertion", "deletion"]`):

* `match`: the most important one, since most notes are matches. A good F-score here means that the notes are correctly aligned.
* `insertion` and `deletion`: these are rare (a good performance has few errors), but they are important for applications that care about performance errors. Since there are few of them, their F-score is sensitive to single errors.

In [ ]:
def fscore_table(prediction: List[dict], ground_truth: List[dict]) -> pd.DataFrame:
    """Precision, recall and F-score of each type (and of all types together)"""
    rows = {}
    for name, types in [
        ("match", ["match"]),
        ("insertion", ["insertion"]),
        ("deletion", ["deletion"]),
        ("all", ["match", "insertion", "deletion"]),
    ]:
        p, r, f, n_pred, n_gt = fscore_alignments(
            prediction, ground_truth, types, return_numbers=True
        )
        rows[name] = {
            "predicted": n_pred, "ground truth": n_gt,
            "precision": p, "recall": r, "F-score": f,
        }
    return pd.DataFrame(rows).T


for m in METHODS:
    print(m)
    display(fscore_table(predictions[m], gt_alignment).round(3))

**Questions**:

* Why is the precision of the `insertion` type so low for the DTW alignments, even though they have a recall of 1?
* Why is the F-score for `deletion` 0 here, if there are no deletions in the ground truth of this performance?
* The F-score for `all` is not the average of the F-scores of the types. Why? (Hint: think about how many pairs of each type there are.)

### Precision vs. Recall: how errors affect the metric

Different errors affect precision and recall differently. Let's simulate two kinds of errors on the ground truth alignment, and see what happens to the metrics:

* **Wrong matches**: a fraction of the matches are given a wrong partner (we rotate the partners of the selected notes). This adds FPs and FNs in the `match` type.
* **Missed matches**: a fraction of the matches are replaced by a `deletion` (of the score note) and an `insertion` (of the performed note), i.e., the algorithm did not find the match, and instead says that the notes were not played/not in the score. This reduces the recall of the `match` type, but it does not add wrong matches.

In [ ]:
def perturb_alignment(
    ground_truth: List[dict], fraction_wrong: float, fraction_missed: float, seed: int = 0
) -> List[dict]:
    rng = np.random.RandomState(seed)
    matches = [a for a in ground_truth if a["label"] == "match"]
    others = [a for a in ground_truth if a["label"] != "match"]
    order = rng.permutation(len(matches))
    n_wrong = int(round(fraction_wrong * len(matches)))
    n_missed = int(round(fraction_missed * len(matches)))
    wrong = [matches[i] for i in order[:n_wrong]]
    missed = [matches[i] for i in order[n_wrong : n_wrong + n_missed]]
    correct = [matches[i] for i in order[n_wrong + n_missed :]]

    # rotate the performed notes of the wrong matches
    partners = [a["performance_id"] for a in wrong]
    partners = partners[1:] + partners[:1]
    wrong = [{**a, "performance_id": p} for a, p in zip(wrong, partners)]
    # missed matches become a deletion and an insertion
    missed_alignment = []
    for a in missed:
        missed_alignment.append({"label": "deletion", "score_id": a["score_id"]})
        missed_alignment.append({"label": "insertion", "performance_id": a["performance_id"]})
    return others + correct + wrong + missed_alignment


def plot_perturbed_metrics(fraction_wrong, fraction_missed):
    perturbed = perturb_alignment(gt_alignment, fraction_wrong, fraction_missed)
    table = fscore_table(perturbed, gt_alignment)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})

    n_tp = len(alignment_to_sets(perturbed)[0] & alignment_to_sets(gt_alignment)[0])
    n_pred = len(alignment_to_sets(perturbed)[0])
    n_gt = len(alignment_to_sets(gt_alignment)[0])
    plot_tp_fp_fn(n_tp, n_pred - n_tp, n_gt - n_tp, ax=axes[0], title="Pairs of type match")

    x = np.arange(len(table))
    for k, metric in enumerate(["precision", "recall", "F-score"]):
        axes[1].bar(x + (k - 1) * 0.27, table[metric], 0.27, label=metric)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(table.index)
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(loc="lower left", ncol=3)
    axes[1].set_title("Metrics by type")
    plt.tight_layout()
    plt.show()


widgets.interact(
    plot_perturbed_metrics,
    fraction_wrong=widgets.FloatSlider(value=0.1, min=0, max=0.5, step=0.05, description="Wrong", continuous_update=False),
    fraction_missed=widgets.FloatSlider(value=0.0, min=0, max=0.5, step=0.05, description="Missed", continuous_update=False),
);

**Questions**:

* What happens to precision and recall of `match` when you only increase the fraction of *wrong* matches? And when you only increase the fraction of *missed* matches?
* Why do the `insertion` and `deletion` F-scores drop so fast when you miss a few matches?

## 5. Comparing Methods and Performances

Finally, let's use the F-score to compare the three alignment methods on all the (short) performances of the dataset. These include two fast and one slow performance, and some contain performance errors (insertions and deletions).

In [ ]:
performance_names = ["short1", "short2", "short_fast1", "short_fast2", "short_slow1"]

results = []
for performance_name in performance_names:
    perf, gt = pt.load_match(
        os.path.join(MATCH_DIR, f"{performance_name}.match"),
        pedal_threshold=127,
        first_note_at_zero=True,
    )
    labels = Counter(a["label"] for a in gt)
    for method in METHODS:
        pred = compute_alignment(method, score, perf)
        table = fscore_table(pred, gt)
        results.append(
            {
                "performance": performance_name,
                "method": method,
                "insertions (gt)": labels["insertion"],
                "deletions (gt)": labels["deletion"],
                **{f"{t} F-score": table.loc[t, "F-score"] for t in ["match", "insertion", "deletion", "all"]},
                "match precision": table.loc["match", "precision"],
                "match recall": table.loc["match", "recall"],
            }
        )

results_df = pd.DataFrame(results)
results_df.set_index(["performance", "method"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
x = np.arange(len(performance_names))
for ax, metric in zip(axes, ["match precision", "match recall", "match F-score"]):
    for k, method in enumerate(METHODS):
        values = results_df[results_df["method"] == method][metric].values
        ax.bar(x + (k - 1) * 0.27, values, 0.27, label=method)
    ax.set_xticks(x)
    ax.set_xticklabels(performance_names, rotation=30)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

results_df.groupby("method")[["match precision", "match recall", "match F-score"]].mean().loc[METHODS].round(3)

**Questions**:

* Which method is best? Is it always the best, in all performances?
* Why does the linear alignment work reasonably well for some performances? What does this tell us about those performances?
* Look at the alignments of the different methods and performances with the plots above (you can change `predictions` and `gt_alignment` to use another performance). What kind of errors do the methods make?

## 6. What the F-score Does (and Does Not) Tell Us

* The F-score is a **note-level, all-or-nothing** metric: a pair is either exactly right or wrong. It does not tell us *how far* a wrong pair is from the right one, and it does not tell us anything about the *timing* of the alignment.
* It is computed **offline**, after the whole performance is available. It cannot tell us how a method behaves *while* the performance is being played (the latency of a score follower, how long it takes to recover from a mistake, etc.).
* It requires a note-level ground truth (like the `.match` files). In practice, many datasets only have onset-level annotations (e.g., beat annotations), and for those other metrics are used.
* Insertions and deletions are rare, so their F-scores can change a lot with a single error. When they are reported, the number of insertions and deletions in the ground truth should be reported too.

In the next part, we will look at metrics for **online alignment** (score following), which look at the *position* error in beats as a function of the time in the performance.

## 7. Online Alignment

In **online alignment** (score following), the algorithm only has access to the performance *up to the current time*, and has to report, as the performance goes on, **where in the score** the performer currently is. The output is therefore a sequence of (performance time, score position in beats) pairs: the *alignment path*, which grows while the performance is played.

Since we are not interested in matching every note, but in *tracking the position*, the metrics are different from the F-score: they measure **how far** (in beats) the reported position is from the true position, at the time of each performed onset.

Matchmaker includes several score followers, as well as the code to evaluate them (`matchmaker.utils.eval`). Let's use four of its followers for MIDI performances:

* `pthmm`: the `PitchHMM` from the `score_follower` notebook (HMM with pitch observations),
* `arzt`: online time warping, in the version by Arzt and Widmer (event-level),
* `dixon`: online time warping, in the version by Dixon (event-level),
* `outerhmm`: an HMM with an observation model based on the outer product of the pitches.

For the evaluation we only need the alignment path of each follower and a ground truth. Let's run them.

In [ ]:
import contextlib
import io
import tempfile

from matchmaker import Matchmaker
from matchmaker.utils.eval import (
    TOLERANCES_IN_BEATS,
    compute_sparc,
    evaluate_alignment,
    resolve_gt,
    spectral_arc_length,
    transfer_positions,
)

MIDI_DIR = os.path.join(MEC_VARIATION_DIR, "performances", "midi")

ONLINE_METHODS = {
    "PitchHMM": "pthmm",
    "OLTW (Arzt)": "arzt",
    "OLTW (Dixon)": "dixon",
    "Outer-product HMM": "outerhmm",
}
ONLINE_COLORS = {
    "PitchHMM": "tab:red",
    "OLTW (Arzt)": "tab:blue",
    "OLTW (Dixon)": "tab:green",
    "Outer-product HMM": "tab:purple",
}


def run_follower(method: str, performance_fn: str, score_fn: str = score_fn) -> np.ndarray:
    """
    Run a Matchmaker score follower on a MIDI performance (the performance is
    "streamed" as if it were played in real time).

    Returns the alignment path: an array with shape (2, T), where the first row is the
    performance time (in seconds) and the second row the position in the score (in beats)
    reported by the follower.
    """
    # Matchmaker is quite verbose
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        mm = Matchmaker(score_fn, performance_fn, input_type="midi", method=method)
        generator = mm.run(verbose=False)
        try:
            while True:
                next(generator)
        except StopIteration as stop:
            path = stop.value
    path = np.array(path, dtype=float)
    return path if path.shape[0] == 2 else path.T


performance_fn = os.path.join(MIDI_DIR, "short1.mid")
paths = {name: run_follower(method, performance_fn) for name, method in ONLINE_METHODS.items()}

print("Alignment path of PitchHMM (performance time in s, position in beats):")
print(paths["PitchHMM"][:, :5])

### Ground truth

The ground truth for online alignment is the **score position (in beats) at the time of each performed onset**. We can get it from the `.match` file with `resolve_gt`. If a chord has several notes, only one position is kept for each position in the score.

In [ ]:
def load_ground_truth(performance_name: str) -> Tuple[np.ndarray, np.ndarray]:
    """Performed onset times (in s) and the corresponding score positions (in beats)"""
    return resolve_gt(
        os.path.join(MATCH_DIR, f"{performance_name}.match"),
        snote_array,
        performance=os.path.join(MIDI_DIR, f"{performance_name}.mid"),
    )


gt_perf_sec, gt_score_beat = load_ground_truth("short1")
print(f"{len(gt_perf_sec)} ground truth onsets")
pd.DataFrame({"performance time (s)": gt_perf_sec, "score position (beats)": gt_score_beat}).head()

In [ ]:
def plot_paths(paths, gt_perf_sec, gt_score_beat, ax=None, xlim=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    for name, path in paths.items():
        ax.step(path[0], path[1], where="post", label=name, color=ONLINE_COLORS[name], linewidth=1.5)
    ax.plot(gt_perf_sec, gt_score_beat, "k.", markersize=5, label="ground truth", zorder=5)
    ax.set_xlabel("Performance time (s)")
    ax.set_ylabel("Score position (beats)")
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.legend()
    return ax


plot_paths(paths, gt_perf_sec, gt_score_beat)
plt.show()

On a clean performance all the followers do very well (the lines lie on top of the ground truth), so there is not much to compare. To see the metrics at work, let's make the problem harder by adding **errors to the performance**: a fraction of the notes are replaced by a wrong pitch, and some random extra notes are added. The true position of the performer in the score does not change, so we can still use the ground truth from the `.match` file.

In [ ]:
from partitura.performance import Performance, PerformedPart

ERROR_DIR = tempfile.mkdtemp()


def make_performance_with_errors(
    performance_name: str, fraction_errors: float, seed: int = 0
) -> str:
    """
    Add errors to a performance: a fraction of the notes get a wrong pitch (a few
    semitones away) and the same fraction of random notes are added.
    Returns the path to the MIDI file of the new performance.
    """
    rng = np.random.RandomState(seed)
    perf = pt.load_performance(os.path.join(MIDI_DIR, f"{performance_name}.mid"))
    notes = [dict(n) for n in perf.performedparts[0].notes]
    for note in notes:
        if rng.rand() < fraction_errors:
            note["pitch"] = int(np.clip(note["pitch"] + rng.choice([-3, -2, -1, 1, 2, 3]), 21, 108))
    t_max = max(n["note_off"] for n in notes)
    for k in range(int(round(fraction_errors * len(notes)))):
        extra = dict(notes[rng.randint(len(notes))])
        onset = rng.uniform(0, t_max - 0.5)
        extra.update(
            pitch=int(np.clip(extra["pitch"] + rng.randint(-12, 13), 21, 108)),
            note_on=onset, note_off=onset + 0.1, sound_off=onset + 0.1, id=f"extra{k}",
        )
        notes.append(extra)
    notes.sort(key=lambda n: n["note_on"])
    fn = os.path.join(ERROR_DIR, f"{performance_name}_errors{fraction_errors:.2f}_{seed}.mid")
    pt.save_performance_midi(Performance(PerformedPart(notes, id="p", part_name="p")), fn)
    return fn


noisy_fn = make_performance_with_errors("short1", fraction_errors=0.1)
noisy_paths = {name: run_follower(method, noisy_fn) for name, method in ONLINE_METHODS.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_paths(noisy_paths, gt_perf_sec, gt_score_beat, ax=axes[0])
axes[0].set_title("Performance with errors")
plot_paths(noisy_paths, gt_perf_sec, gt_score_beat, ax=axes[1], xlim=(0, 6))
axes[1].set_title("Zoom")
plt.tight_layout()
plt.show()

Now the followers do make mistakes (some are small, and some are large), and we need metrics to describe them.

## 8. Metrics for Online Alignment

### 8.1 The beat error

The basic quantity is the **beat error**. For each onset $i$ of the ground truth, with performance time $t_i$ and true score position $b_i$:

1. We look up the position that the follower **had reported at time $t_i$**: $\hat{b}_i$ (the last position that it reported up to $t_i$; this is what `transfer_positions` does).
2. The beat error is $e_i = b_i - \hat{b}_i$ (in beats). If $e_i > 0$ the follower is *behind* the performer, and if $e_i < 0$ it is *ahead*.

Let's see this for the Outer-product HMM. The black dots are the ground truth, the line is the position reported by the follower, and the red segments are the beat errors:

In [ ]:
def beat_errors(path: np.ndarray, gt_perf_sec: np.ndarray, gt_score_beat: np.ndarray):
    """Positions reported by the follower at the time of the ground truth onsets, and the signed beat errors"""
    predicted = transfer_positions(path, gt_perf_sec, frame_rate=1, domain="score")
    return predicted, gt_score_beat - predicted


def plot_beat_errors(path, gt_perf_sec, gt_score_beat, window=4.0, ax=None):
    predicted, errors = beat_errors(path, gt_perf_sec, gt_score_beat)
    # zoom in on the largest error
    center = gt_perf_sec[np.nanargmax(np.abs(errors))]
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    ax.step(path[0], path[1], where="post", color="0.3", label="position reported by the follower")
    ax.plot(gt_perf_sec, gt_score_beat, "k.", markersize=7, label="ground truth", zorder=5)
    ax.vlines(gt_perf_sec, predicted, gt_score_beat, color="tab:red", linewidth=2, label="beat error", zorder=4)
    ax.set_xlim(center - window / 2, center + window / 2)
    in_window = (gt_perf_sec > center - window / 2) & (gt_perf_sec < center + window / 2)
    ax.set_ylim(np.nanmin(np.r_[predicted[in_window], gt_score_beat[in_window]]) - 0.5,
                np.nanmax(np.r_[predicted[in_window], gt_score_beat[in_window]]) + 0.5)
    ax.set_xlabel("Performance time (s)")
    ax.set_ylabel("Score position (beats)")
    ax.legend(loc="lower right")
    return ax


plot_beat_errors(noisy_paths["Outer-product HMM"], gt_perf_sec, gt_score_beat)
plt.show()

Matchmaker summarizes the beat errors of the whole performance with several statistics. Let's compute them by hand for this follower, and compare them with the output of `evaluate_alignment`:

In [ ]:
path = noisy_paths["Outer-product HMM"]
predicted, errors = beat_errors(path, gt_perf_sec, gt_score_beat)

print(f"mean absolute error:   {np.nanmean(np.abs(errors)):.4f} beats")
print(f"median absolute error: {np.nanmedian(np.abs(errors)):.4f} beats")
print(f"std of absolute error: {np.nanstd(np.abs(errors)):.4f} beats")

results = evaluate_alignment(
    aligned_score_beats=path[1],
    aligned_perf_sec=path[0],
    gt_score_beats=gt_score_beat,
    gt_perf_sec=gt_perf_sec,
)
assert np.isclose(results["mean_b"], np.nanmean(np.abs(errors)), atol=1e-4)
pd.Series(results, name="Outer-product HMM").to_frame()

`evaluate_alignment` returns:

* `mean_b`, `median_b`, `std_b`: mean, median and standard deviation of the **absolute** beat error. The median is robust to a few large errors (e.g., when the follower gets lost for a moment), while the mean is not: if the median is small and the mean is large, the follower is mostly right, but sometimes very wrong.
* `skewness`, `kurtosis`: shape of the distribution of the **signed** beat error. A skewness different from 0 means that the follower tends to be late (positive) or early (negative) when it makes errors. A large kurtosis means that errors are concentrated in a few large outliers (heavy tails). They are not defined (`NaN`) if there are no errors at all.
* `accuracy_<tolerance>b`: proportion of onsets for which the absolute beat error is at most a given tolerance (0.05, 0.1, 0.3, 0.5, 1 or 2 beats).
* `smoothness`: how smooth the reported trajectory is (see below).

Let's look at the distribution of the signed errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

# Signed beat errors over time
for name, p in noisy_paths.items():
    _, err = beat_errors(p, gt_perf_sec, gt_score_beat)
    axes[0].plot(gt_perf_sec, err, label=name, color=ONLINE_COLORS[name], linewidth=1.2)
axes[0].axhline(0, color="k", linewidth=0.8)
axes[0].set_xlabel("Performance time (s)")
axes[0].set_ylabel("Signed beat error (beats)\n(> 0: follower behind)")
axes[0].legend()

# Histogram of the signed errors of the Outer-product HMM
bins = np.linspace(-3, 3, 61)
axes[1].hist(np.clip(errors, -3, 3), bins=bins, color=ONLINE_COLORS["Outer-product HMM"])
axes[1].set_yscale("log")
axes[1].set_xlabel("Signed beat error (beats, clipped to ±3), Outer-product HMM")
axes[1].set_ylabel("Number of onsets (log scale)")
axes[1].set_title(f"skewness = {results['skewness']:.2f}, kurtosis = {results['kurtosis']:.2f}")
plt.tight_layout()
plt.show()

**Question**: Most of the errors are exactly 0, and a few are large. Which of the statistics (mean, median, skewness, kurtosis) capture this?

### 8.2 Accuracy at different tolerances

Instead of summarizing the errors with a mean, we can ask: *for which proportion of the onsets is the follower within a given tolerance?* Different applications need different tolerances: for page turning, a couple of beats is enough, while for an accompaniment system we need a very precise position. Matchmaker reports the accuracy for tolerances of 0.05, 0.1, 0.3, 0.5, 1 and 2 beats.

This is the (empirical) cumulative distribution of the absolute beat error, evaluated at the tolerances:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, p in noisy_paths.items():
    _, err = beat_errors(p, gt_perf_sec, gt_score_beat)
    abs_err = np.sort(np.abs(err[~np.isnan(err)]))
    ax.step(abs_err, np.arange(1, len(abs_err) + 1) / len(gt_perf_sec), where="post",
            label=name, color=ONLINE_COLORS[name])
for tau in TOLERANCES_IN_BEATS:
    ax.axvline(tau, color="0.7", linestyle=":", zorder=0)
ax.set_xscale("symlog", linthresh=0.05)
ax.set_xticks([0] + TOLERANCES_IN_BEATS)
ax.set_xticklabels(["0"] + [f"{t:g}" for t in TOLERANCES_IN_BEATS])
ax.set_xlabel("Tolerance (beats)")
ax.set_ylabel("Accuracy (proportion of onsets\nwith error $\\leq$ tolerance)")
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

A follower is better if its curve is higher (more onsets within the tolerance). The accuracy at a given tolerance does not tell us how large the errors that exceed the tolerance are, but the curves do.

### 8.3 Smoothness

A follower that reports the correct position on average can still be *unusable* for applications like automatic accompaniment or page turning if its position jumps back and forth. Matchmaker measures this with the **spectral arc length** (SPARC) of the trajectory (Balasubramanian et al., 2015), a measure of movement smoothness:

1. The reported position is resampled at a fixed rate (50 Hz), holding the last reported position between updates.
2. The **speed** of the follower (how fast the position changes) is computed.
3. We take the magnitude of the Fourier transform of the speed profile (normalized to a maximum of 1), and compute the **arc length of this curve** (the length of the curve in the plot, with the frequency axis normalized).

A smooth movement has a speed profile with a simple spectrum (a short curve), while a jerky one (with sudden jumps in the position) has a lot of high-frequency content (a long curve). The smoothness is the **negative** arc length, so that values closer to 0 are smoother. This is computed in windows (of 30 s) that start at the performed onsets, and averaged. Our performance is shorter than the window, so there is a single window.

Let's see how this looks for a smooth and for a jerky follower:

In [ ]:
def speed_profile(path: np.ndarray, gt_perf_sec: np.ndarray, fs: float = 50.0):
    """Speed of the reported position (resampled at `fs` Hz) in the span of the ground truth onsets"""
    times = np.arange(gt_perf_sec.min(), gt_perf_sec.max(), 1.0 / fs)
    idx = np.searchsorted(path[0], times, side="right") - 1
    position = path[1][np.clip(idx, 0, None)]
    return times, position, np.abs(np.gradient(position, 1.0 / fs))


fig, axes = plt.subplots(2, 3, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
for row, name in enumerate(["PitchHMM", "Outer-product HMM"]):
    times, position, speed = speed_profile(noisy_paths[name], gt_perf_sec)
    n_fft = int(2 ** (np.ceil(np.log2(len(speed))) + 4))
    spectrum = np.abs(np.fft.rfft(speed, n=n_fft))
    spectrum /= spectrum.max()
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / 50.0)
    mask = freqs <= 10
    axes[row, 0].plot(times, position, color=ONLINE_COLORS[name])
    axes[row, 0].set_ylabel(f"{name}\nposition (beats)")
    axes[row, 1].plot(times, speed, color=ONLINE_COLORS[name])
    axes[row, 1].set_ylabel("speed (beats/s)")
    axes[row, 2].plot(freqs[mask], spectrum[mask], color=ONLINE_COLORS[name])
    axes[row, 2].set_ylabel("normalized spectrum")
    axes[row, 2].set_title(f"SPARC = {spectral_arc_length(speed, 50.0):.2f}")
for ax in axes[1]:
    ax.set_xlabel("Performance time (s)" if ax is not axes[1, 2] else "Frequency (Hz)")
axes[0, 0].set_title("Position")
axes[0, 1].set_title("Speed")
plt.tight_layout()
plt.show()

`compute_sparc` does this for all the windows, and `evaluate_alignment` calls it to compute `smoothness`. Since the smoothness depends on the *tempo* of the performance (a fast performance has a faster change in position), it should only be compared between followers on the **same performance**.

### 8.4 All metrics together

Let's put all the metrics together for the four followers on the performance with errors:

In [ ]:
def evaluate_online(path: np.ndarray, gt_perf_sec: np.ndarray, gt_score_beat: np.ndarray) -> dict:
    return evaluate_alignment(
        aligned_score_beats=path[1],
        aligned_perf_sec=path[0],
        gt_score_beats=gt_score_beat,
        gt_perf_sec=gt_perf_sec,
    )


pd.DataFrame(
    {name: evaluate_online(p, gt_perf_sec, gt_score_beat) for name, p in noisy_paths.items()}
).round(3)

**Questions**:

* Which follower is best according to the mean error? And according to the median? Why can these two disagree?
* Which follower is the *smoothest*? Is it also the most accurate? Can you explain why smoothness and accuracy are different things?
* The tolerance that we need depends on the application. Which follower would you choose for page turning? And for an automatic accompaniment system?

## 9. Interactive Exploration: Robustness to Errors

Use the sliders to change the fraction of errors in the performance, and compare the followers:

In [ ]:
def plot_online_comparison(performance_name, fraction_errors, seed):
    if fraction_errors == 0:
        fn = os.path.join(MIDI_DIR, f"{performance_name}.mid")
    else:
        fn = make_performance_with_errors(performance_name, fraction_errors, seed=seed)
    gt_t, gt_b = load_ground_truth(performance_name)
    local_paths = {name: run_follower(method, fn) for name, method in ONLINE_METHODS.items()}
    table = pd.DataFrame({name: evaluate_online(p, gt_t, gt_b) for name, p in local_paths.items()})

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1.4, 1, 1]})
    for name, p in local_paths.items():
        _, err = beat_errors(p, gt_t, gt_b)
        axes[0].plot(gt_t, err, color=ONLINE_COLORS[name], label=name, linewidth=1.2)
    axes[0].axhline(0, color="k", linewidth=0.8)
    axes[0].set_xlabel("Performance time (s)")
    axes[0].set_ylabel("Signed beat error (beats)")
    axes[0].legend(fontsize=8)

    x = np.arange(len(TOLERANCES_IN_BEATS))
    for k, name in enumerate(local_paths):
        acc = [table.loc[f"accuracy_{t:g}b", name] for t in TOLERANCES_IN_BEATS]
        axes[1].plot(x, acc, marker="o", color=ONLINE_COLORS[name], label=name)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{t:g}" for t in TOLERANCES_IN_BEATS])
    axes[1].set_xlabel("Tolerance (beats)")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1.02)

    axes[2].bar(range(len(local_paths)), table.loc["smoothness"].astype(float),
                color=[ONLINE_COLORS[n] for n in local_paths])
    axes[2].set_xticks(range(len(local_paths)))
    axes[2].set_xticklabels(list(local_paths), rotation=30, ha="right")
    axes[2].set_ylabel("Smoothness (SPARC)")
    plt.tight_layout()
    plt.show()
    display(table.round(3))


widgets.interact(
    plot_online_comparison,
    performance_name=widgets.Dropdown(options=performance_names, value="short1", description="Performance"),
    fraction_errors=widgets.FloatSlider(value=0.1, min=0, max=0.3, step=0.05, description="Errors", continuous_update=False),
    seed=widgets.IntSlider(value=0, min=0, max=9, description="Seed", continuous_update=False),
);

## 10. Comparing Followers on All Performances

Finally, let's evaluate the four followers on all the short performances of the dataset, both as they are (*clean*) and with 10% errors.

In [ ]:
online_results = []
for performance_name in performance_names:
    gt_t, gt_b = load_ground_truth(performance_name)
    for condition, fraction in [("clean", 0.0), ("10% errors", 0.1)]:
        fn = (
            os.path.join(MIDI_DIR, f"{performance_name}.mid")
            if fraction == 0
            else make_performance_with_errors(performance_name, fraction)
        )
        for name, method in ONLINE_METHODS.items():
            res = evaluate_online(run_follower(method, fn), gt_t, gt_b)
            online_results.append({"performance": performance_name, "condition": condition, "follower": name, **res})

online_df = pd.DataFrame(online_results)
metrics = ["mean_b", "median_b", "accuracy_0.05b", "accuracy_0.3b", "accuracy_1b", "smoothness"]
online_df.groupby(["condition", "follower"])[metrics].mean().loc[["clean", "10% errors"]].round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
x = np.arange(len(performance_names))
for row, condition in enumerate(["clean", "10% errors"]):
    for ax, metric in zip(axes[row], ["mean_b", "accuracy_0.3b", "smoothness"]):
        for k, name in enumerate(ONLINE_METHODS):
            values = online_df[(online_df["condition"] == condition) & (online_df["follower"] == name)][metric]
            ax.bar(x + (k - 1.5) * 0.2, values.astype(float), 0.2, label=name, color=ONLINE_COLORS[name])
        ax.set_xticks(x)
        ax.set_xticklabels(performance_names, rotation=30)
        ax.set_title(f"{metric} ({condition})")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Questions**:

* On the clean performances, are the differences between the followers visible in all metrics? Which metric is the most informative here?
* How do the errors affect each follower? Do the followers that are affected the most have a particular kind of error (look at the plots above)?

## 11. Summary: Offline vs. Online Metrics

| | Offline (F-score) | Online (Matchmaker) |
|---|---|---|
| What is evaluated | pairs of notes (score note, performed note) | the score position (beats) reported at each performed onset |
| Ground truth | note-level alignment (`match`, `insertion`, `deletion`) | position of each performed onset in the score |
| Error | a pair is right or wrong | beat error: *how far* the position is from the truth |
| Summary | precision, recall, F-score per type | mean/median/std of the error, accuracy at tolerances, skewness/kurtosis |
| Temporal behavior | not measured | smoothness of the trajectory (SPARC) |

Note that the online metrics evaluate only the position at the onsets of the ground truth. They do not tell us anything about the notes that the follower considers to be wrong notes or missing notes, and, as computed here (with a simulated stream), they do not include the processing delay of a real-time system.